<a href="https://colab.research.google.com/github/NickoGran/cloud-provider-analytics/blob/main/notebooks/01_evaluacion_1_fundacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto Integrador — Minería de Datos II

## Cloud Provider Analytics

**Institución:** ISTEA  
**Asignatura:** Minería de Datos II  
**Proyecto:** Cloud Provider Analytics  
**Evaluación:** Primera evaluación parcial — Diseño y fundación de datos  
**Tecnología principal:** Apache Spark con PySpark  
**Entorno de trabajo:** Google Colab  

---

## Objetivo de esta primera entrega

El objetivo de esta primera etapa es comprender el problema planteado, analizar las fuentes de datos provistas y diseñar una arquitectura viable para una solución de datos que posteriormente será implementada mediante procesamiento batch y streaming.

La solución deberá contemplar la ingestión de diferentes fuentes, la organización de un Data Lake, el procesamiento con Apache Spark y la posterior publicación de información preparada para los dominios analíticos de FinOps, Soporte y Producto/Usage.

En esta primera evaluación no se busca implementar todavía el pipeline end-to-end completo, sino establecer una fundación técnica coherente, justificable y reproducible para las siguientes etapas del proyecto.

---

## Alcance de la primera evaluación

Durante este notebook se resolverán los 12 puntos solicitados para la primera evaluación:

1. Interpretación del problema, usuarios, preguntas principales y objetivos medibles.
2. Justificación de la necesidad de Big Data mediante las 5V: volumen, velocidad, variedad, veracidad y valor.
3. Inventario y perfil inicial de las fuentes: grano, frecuencia, tipos, calidad, trazabilidad y riesgos.
4. Diseño de una arquitectura de alto nivel con fuentes, ingesta, Data Lake, procesamiento, serving, consumo y capacidades transversales.
5. Selección y justificación del patrón arquitectónico.
6. Mapeo entre requisitos y componentes de la arquitectura.
7. Diseño del Data Lake: zonas, formatos, particiones, naming, retención, metadatos y reglas de promoción.
8. Diseño de los flujos batch y streaming desde las fuentes hasta las salidas.
9. Representación de un flujo batch mediante lógica MapReduce o equivalente conceptual.
10. Identificación de supuestos, riesgos, mitigaciones y decisiones abiertas.
11. Estimación preliminar de esfuerzo, roles y recursos.
12. Fundación del repositorio y generación de evidencia mínima de lectura y exploración de los datos.

---

## Criterio metodológico

Para mantener la solución alineada con los contenidos de la cursada, se priorizarán las herramientas y conceptos trabajados en clase y los expresamente requeridos por la consigna.

Se utilizarán principalmente:

- Google Colab.
- Python.
- Apache Spark mediante PySpark.
- DataFrames.
- CSV, JSON/JSONL y Parquet.
- Procesamiento batch.
- Structured Streaming.
- Transformaciones y acciones de Spark.
- Particiones y shuffle.
- Joins y agregaciones.
- Lógica MapReduce.
- Arquitectura de datos con zonas Landing, Bronze, Silver y Gold.

Se evitará incorporar tecnología adicional que no sea necesaria para resolver los requisitos del proyecto.

# 0 - Preparación del entorno

Antes de comenzar el análisis se configura Apache Spark en Google Colab.

Se utilizará una ejecución local de Spark sobre la máquina virtual proporcionada por Colab. Esto permite desarrollar y validar la lógica del procesamiento utilizando la misma API que posteriormente puede ejecutarse en un entorno distribuido.

`SparkSession` será el punto de entrada principal para trabajar con DataFrames y realizar las operaciones de lectura, transformación y análisis.

In [25]:
!pip -q install pyspark==4.0.4

## 0.1 Creación de la sesión de Spark

Se crea una `SparkSession`, que constituye el punto de entrada principal para trabajar con Apache Spark.

Se utilizará:

- `local[*]` para aprovechar los núcleos disponibles en la máquina virtual de Google Colab.
- `spark.sql.shuffle.partitions = 8` como configuración inicial para las operaciones que requieran redistribución de datos.

Esta configuración es suficiente para trabajar con el dataset reducido provisto para el proyecto.

In [26]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("cloud-provider-analytics")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Versión de Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)

Versión de Spark: 4.0.4
Master: local[*]
Shuffle partitions: 8


## 0.2 Carga del dataset provisto

El proyecto utiliza exclusivamente el dataset provisto para el caso **Cloud Provider Analytics**.

El archivo comprimido contiene las fuentes originales dentro de la zona `datalake/landing`.

Los datos de Landing representan los archivos recibidos desde los sistemas de origen y deben considerarse datos crudos e inmutables. Por lo tanto, durante el proyecto no se modificarán los archivos originales.

Para trabajar en Google Colab se cargará y descomprimirá una copia del dataset dentro del entorno temporal de ejecución.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from pathlib import Path
import zipfile

# Archivo ZIP cargado en Google Colab
ZIP_PATH = Path("/content/cloud_provider_challenge_dataset_v1.zip")

# Carpeta de trabajo
DATASET_PATH = Path("/content/cloud_provider_challenge_dataset_v1")

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        "No se encontró cloud_provider_challenge_dataset_v1.zip. "
        "Verificar que el archivo haya sido cargado correctamente."
    )

DATASET_PATH.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(DATASET_PATH)

LANDING_PATH = DATASET_PATH / "datalake" / "landing"

print("Dataset descomprimido correctamente.")
print("Ruta Landing:", LANDING_PATH)

In [ ]:
print("Contenido de Landing:\n")

for path in sorted(LANDING_PATH.iterdir()):
    print("-", path.name)

# 1 - Interpretación del problema, usuarios, preguntas y objetivos

## 1.1 Contexto del problema

El proyecto representa al área de datos de un proveedor de servicios cloud.

La organización dispone de información proveniente de diferentes dominios: clientes, usuarios, recursos cloud, soporte, marketing, encuestas NPS, facturación y eventos de utilización de los servicios.

Estas fuentes presentan diferentes características y frecuencias de actualización. Parte de la información puede procesarse mediante cargas batch, mientras que los eventos de utilización requieren una capacidad de procesamiento cercana al tiempo real.

El objetivo general es construir una arquitectura de datos capaz de ingestar, organizar, limpiar, conformar y publicar esta información para permitir su utilización por diferentes áreas de la organización.

La solución deberá permitir posteriormente implementar un flujo que contemple:

**Fuentes → Ingesta → Data Lake → Procesamiento → Serving → Consumo**

y que incorpore de forma transversal aspectos de calidad, trazabilidad, gobierno, seguridad y observabilidad.

---

## 1.2 Usuarios y dominios analíticos

Se identifican tres dominios principales de consumo.

### FinOps

Necesita analizar:

- costos;
- consumo;
- revenue;
- créditos;
- impuestos;
- anomalías;
- eficiencia por organización y servicio.

### Soporte

Necesita analizar:

- volumen de tickets;
- severidad;
- cumplimiento de SLA;
- satisfacción del cliente;
- evolución de los indicadores por organización y fecha.

### Producto / Usage

Necesita analizar:

- utilización de los servicios cloud;
- cantidad de requests;
- métricas operativas;
- utilización de GenAI;
- tokens consumidos;
- emisiones de carbono cuando la información esté disponible.

---

## 1.3 Preguntas principales

La arquitectura propuesta deberá permitir responder progresivamente preguntas como:

1. ¿Cuál es el costo diario de cada organización por servicio?
2. ¿Cuántas requests genera cada organización y servicio?
3. ¿Cuáles son los servicios con mayor costo acumulado?
4. ¿Existen comportamientos anómalos o incrementos inusuales de costos?
5. ¿Cómo evoluciona el volumen de tickets de soporte?
6. ¿Qué porcentaje de los tickets incumple el SLA?
7. ¿Cuál es el nivel de satisfacción obtenido mediante CSAT y NPS?
8. ¿Cuál es el revenue mensual de cada organización considerando créditos, impuestos y moneda?
9. ¿Cuántos tokens GenAI utiliza cada organización cuando dicha métrica se encuentra disponible?
10. ¿Qué información de utilización y costos puede ser presentada posteriormente a herramientas analíticas?

---

## 1.4 Objetivo general

Diseñar la fundación de una solución de datos reproducible que permita integrar las fuentes provistas, diferenciando correctamente los procesos batch de los eventos que requieren procesamiento streaming, y organizar la información en un Data Lake preparado para las siguientes etapas del proyecto.

---

## 1.5 Objetivos medibles de la primera etapa

Para considerar cumplida esta primera etapa se buscará:

- identificar y documentar el 100 % de las fuentes provistas;
- determinar el grano y las principales características de calidad de cada fuente;
- diferenciar las fuentes que serán tratadas mediante batch de la fuente de eventos destinada a streaming;
- verificar mediante exploración reproducible los principales problemas de calidad presentes en los datos;
- diseñar las zonas Landing, Bronze, Silver y Gold del Data Lake;
- definir los formatos y criterios iniciales de particionamiento;
- diseñar una arquitectura que permita responder los dominios FinOps, Soporte y Producto/Usage;
- establecer trazabilidad entre los requisitos del caso y los componentes de la arquitectura;
- identificar riesgos, supuestos y decisiones pendientes antes de comenzar la implementación end-to-end.

# 2 - Justificación de Big Data mediante las 5V

La necesidad de una arquitectura de Big Data será evaluada mediante cinco dimensiones:

- **Volumen:** cantidad de información a procesar y posibilidad de crecimiento.
- **Velocidad:** frecuencia con la que los datos son generados o deben procesarse.
- **Variedad:** diversidad de fuentes, formatos, estructuras y dominios.
- **Veracidad:** calidad, consistencia y confiabilidad de los datos.
- **Valor:** capacidad de transformar los datos en información útil para los usuarios del negocio.

Siempre que sea posible, estas dimensiones se respaldarán con evidencia obtenida directamente del dataset provisto.

Es importante distinguir entre las características observables en el dataset académico y las características que representan el escenario productivo simulado. El dataset de la cursada es deliberadamente reducido y los archivos de eventos simulan micro-lotes; por lo tanto, no corresponde afirmar que el volumen actual sea masivo ni medir una velocidad productiva inexistente. La justificación deberá considerar tanto la evidencia disponible como el comportamiento que el caso busca representar.

## 2.1 Evidencia de volumen

Primero se cuantificarán las fuentes disponibles y la cantidad de registros contenidos en cada una.

El objetivo no es demostrar que el dataset académico posee un volumen masivo, sino establecer una línea base cuantitativa y analizar cómo se distribuye la información entre las distintas fuentes.

In [ ]:
from pathlib import Path

# Fuentes CSV
csv_paths = sorted(LANDING_PATH.glob("*.csv"))

# Fragmentos JSONL de eventos
stream_path = LANDING_PATH / "usage_events_stream"
jsonl_paths = sorted(stream_path.glob("*.jsonl"))

inventario = []

# Conteo de registros de cada CSV utilizando Spark
for path in csv_paths:
    df = (
        spark.read
        .option("header", True)
        .csv(str(path))
    )

    inventario.append(
        (
            path.name,
            "CSV",
            df.count(),
            path.stat().st_size
        )
    )

# Lectura conjunta de los eventos JSONL
events = spark.read.json(str(stream_path / "*.jsonl"))

inventario.append(
    (
        "usage_events_stream/*.jsonl",
        "JSONL",
        events.count(),
        sum(path.stat().st_size for path in jsonl_paths)
    )
)

inventario_df = spark.createDataFrame(
    inventario,
    ["fuente", "formato", "registros", "bytes"]
)

inventario_df.show(truncate=False)

print("Cantidad de archivos JSONL:", len(jsonl_paths))

## 2.2 Evidencia de velocidad

La fuente `usage_events_stream` fue provista de forma fragmentada para representar el comportamiento de micro-lotes.

Se analizará:

- cantidad de fragmentos;
- cantidad total de eventos;
- fecha mínima y máxima de los eventos;
- versiones de esquema presentes.

La existencia de archivos fragmentados no constituye por sí misma un streaming real. El dataset es estático y simula el ingreso progresivo de eventos. La velocidad constituye, por lo tanto, un requisito arquitectónico del caso que posteriormente será implementado mediante Structured Streaming.

In [ ]:
from pyspark.sql import functions as F

events_temporal = (
    events
    .withColumn(
        "event_timestamp",
        F.to_timestamp("timestamp")
    )
)

resumen_stream = (
    events_temporal
    .agg(
        F.count("*").alias("cantidad_eventos"),
        F.countDistinct("event_id").alias("eventos_unicos"),
        F.min("event_timestamp").alias("fecha_minima"),
        F.max("event_timestamp").alias("fecha_maxima"),
        F.countDistinct("schema_version").alias("versiones_schema")
    )
)

resumen_stream.show(truncate=False)

print("Cantidad de micro-lotes simulados:", len(jsonl_paths))

## 2.3 Evidencia de variedad

La variedad será analizada considerando:

1. la existencia de diferentes formatos de entrada;
2. los diferentes dominios de negocio;
3. la diversidad de servicios, métricas y regiones contenidas en los eventos.

In [ ]:
print("Servicios presentes:")
(
    events
    .select("service")
    .distinct()
    .orderBy("service")
    .show(truncate=False)
)

print("Métricas presentes:")
(
    events
    .select("metric")
    .distinct()
    .orderBy("metric")
    .show(truncate=False)
)

print("Regiones presentes:")
(
    events
    .select("region")
    .distinct()
    .orderBy("region")
    .show(truncate=False)
)

print("Versiones de esquema:")
(
    events
    .groupBy("schema_version")
    .count()
    .orderBy("schema_version")
    .show()
)

## 2.4 Evidencia de veracidad

La veracidad se relaciona con la calidad y confiabilidad de los datos.

La consigna anticipa problemas como:

- valores nulos;
- tipos ambiguos;
- costos negativos;
- valores atípicos;
- evolución del esquema.

Estos problemas serán comprobados directamente sobre los datos antes de definir las reglas de calidad y limpieza que posteriormente se implementarán en las capas Bronze y Silver.

In [ ]:
print("Distribución por versión de esquema:")

(
    events
    .groupBy("schema_version")
    .count()
    .orderBy("schema_version")
    .show()
)

print("Eventos con value nulo:")

events.filter(
    F.col("value").isNull()
).select(
    F.count("*").alias("cantidad")
).show()

print("Eventos con unit nulo:")

events.filter(
    F.col("unit").isNull()
).select(
    F.count("*").alias("cantidad")
).show()

print("Eventos con costo negativo:")

events.filter(
    F.col("cost_usd_increment") < 0
).select(
    F.count("*").alias("cantidad")
).show()

### Comprobación de tipos ambiguos en el JSON original

Cuando Spark lee conjuntamente archivos JSON que contienen diferentes tipos para una misma columna, puede seleccionar un tipo común para representar la columna completa.

Por ese motivo, para comprobar si el campo `value` llega originalmente algunas veces como número y otras veces como texto, se inspeccionarán directamente los valores del JSON crudo.

Esta comprobación se realiza únicamente sobre los archivos de Landing, sin modificarlos.

In [ ]:
import json

tipos_value = {
    "numerico": 0,
    "texto": 0,
    "nulo": 0,
    "otro": 0
}

for path in jsonl_paths:
    with open(path, "r", encoding="utf-8") as archivo:

        for linea in archivo:
            evento = json.loads(linea)
            value = evento.get("value")

            if value is None:
                tipos_value["nulo"] += 1

            elif isinstance(value, str):
                tipos_value["texto"] += 1

            elif isinstance(value, (int, float)):
                tipos_value["numerico"] += 1

            else:
                tipos_value["otro"] += 1

print("Tipos originales encontrados en value:")

for tipo, cantidad in tipos_value.items():
    print(f"{tipo}: {cantidad}")

## 2.5 Evolución del esquema

Los eventos incluyen un campo `schema_version`.

La versión 2 incorpora nueva información, incluyendo `carbon_kg` y, para eventos asociados al servicio GenAI, `genai_tokens`.

Se verificará cuándo aparece cada versión para confirmar que el pipeline deberá ser capaz de compatibilizar diferentes versiones del esquema.

In [ ]:
(
    events_temporal
    .groupBy("schema_version")
    .agg(
        F.count("*").alias("cantidad"),
        F.min("event_timestamp").alias("desde"),
        F.max("event_timestamp").alias("hasta")
    )
    .orderBy("schema_version")
    .show(truncate=False)
)

## 2.6 Interpretación de las 5V

### Volumen

El dataset académico contiene ocho fuentes lógicas y, dentro de la fuente de eventos, 43.200 registros distribuidos en 120 archivos JSONL.

Este volumen puede procesarse actualmente en una única máquina y no constituye por sí solo un caso de Big Data a escala productiva.

Sin embargo, `usage_events_stream` representa una fuente cuyo volumen aumenta continuamente a medida que nuevas organizaciones, recursos, servicios y eventos son incorporados. Por este motivo, la arquitectura debe diseñarse considerando crecimiento y procesamiento distribuido, aunque la muestra utilizada para la evaluación sea deliberadamente reducida.

---

### Velocidad

Los 43.200 eventos se encuentran fragmentados en 120 archivos JSONL que representan micro-lotes y cubren aproximadamente 60 días.

El dataset entregado es estático, por lo que no permite medir una tasa real de eventos por segundo ni una latencia productiva.

Sin embargo, el caso requiere procesamiento near real-time de eventos de utilización. Esta característica justifica que el diseño contemple Structured Streaming para `usage_events_stream`, mientras que las fuentes maestras y de facturación pueden ser procesadas mediante batch.

---

### Variedad

Existe variedad tanto en formatos como en dominios.

Los datos de Landing incluyen:

- archivos CSV para clientes, usuarios, recursos, soporte, marketing, NPS y facturación;
- archivos JSONL para eventos de utilización.

Además, los eventos corresponden a diferentes servicios cloud, entre ellos compute, storage, database, networking, analytics y genai, y contienen diferentes métricas operativas.

Esta diversidad requiere procesos de estandarización y conformado antes de realizar análisis conjuntos.

---

### Veracidad

La exploración inicial confirma la existencia de problemas de calidad y consistencia.

Entre los eventos analizados se detectaron:

- 877 registros con `value` nulo;
- 2.075 registros con `unit` nulo;
- 216 registros con costos negativos;
- 1.309 valores de `value` almacenados originalmente como texto;
- dos versiones diferentes del esquema.

La versión 1 contiene 10.800 eventos y la versión 2 contiene 32.400 eventos.

A partir del 18/07/2025 aparece la segunda versión del esquema, incorporando información adicional como `carbon_kg` y `genai_tokens` cuando corresponde.

Estos resultados justifican la necesidad de tipificación explícita, validaciones, tratamiento de nulos, control de anomalías y compatibilización de esquemas.

---

### Valor

La información disponible puede transformarse en productos analíticos destinados a tres dominios principales:

- **FinOps:** costos, consumo, revenue, créditos, impuestos y anomalías.
- **Soporte:** tickets, severidad, SLA y satisfacción.
- **Producto/Usage:** utilización de servicios, requests, métricas operativas, GenAI y carbono.

A diferencia de las dimensiones anteriores, el valor no se demuestra únicamente mediante cantidad de registros. Surge de la capacidad de integrar y transformar las diferentes fuentes para responder preguntas de negocio útiles.

---

## Conclusión de las 5V

La justificación del uso de tecnologías de Big Data no se basa en afirmar que el dataset académico actual sea masivo.

La necesidad surge principalmente de la combinación de:

- crecimiento esperado del volumen de eventos;
- necesidad de procesamiento batch y near real-time;
- diversidad de fuentes y formatos;
- problemas reales de calidad y evolución de esquema;
- necesidad de producir información reutilizable para distintos dominios del negocio.

Por lo tanto, Apache Spark resulta apropiado para diseñar una solución reproducible que combine procesamiento batch, procesamiento incremental y preparación de datos analíticos.

# 3 - Inventario y perfil inicial de las fuentes

En esta sección se realizará un análisis inicial de todas las fuentes disponibles en Landing.

El objetivo es conocer cada fuente antes de diseñar sus transformaciones y definir:

- formato;
- cantidad de registros;
- columnas;
- grano;
- clave candidata;
- comportamiento temporal;
- modalidad de ingesta esperada;
- valores nulos;
- duplicados;
- integridad referencial;
- trazabilidad;
- principales riesgos de calidad.

Esta exploración se realiza sobre los datos crudos sin modificar los archivos de Landing.

La intención no es realizar todavía la limpieza definitiva de los datos. Las reglas de tipificación, normalización y conformado serán responsabilidad de las siguientes capas del Data Lake.

## 3.1 Carga lógica de las fuentes para exploración

Los archivos CSV serán leídos inicialmente sin inferencia automática de tipos.

Esta decisión permite observar los datos de Landing sin asumir todavía una tipificación definitiva. Los esquemas explícitos que utilizará el pipeline serán definidos posteriormente durante el diseño e implementación de Bronze.

La fuente de eventos ya fue leída previamente desde los archivos JSONL para realizar el análisis de las 5V.

In [ ]:
# Rutas de las fuentes CSV disponibles en Landing

fuentes_csv = {
    "customers_orgs": LANDING_PATH / "customers_orgs.csv",
    "users": LANDING_PATH / "users.csv",
    "resources": LANDING_PATH / "resources.csv",
    "support_tickets": LANDING_PATH / "support_tickets.csv",
    "marketing_touches": LANDING_PATH / "marketing_touches.csv",
    "nps_surveys": LANDING_PATH / "nps_surveys.csv",
    "billing_monthly": LANDING_PATH / "billing_monthly.csv"
}


# Lectura de los CSV como datos crudos

dataframes = {}

for nombre, ruta in fuentes_csv.items():

    dataframes[nombre] = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(str(ruta))
    )


# Incorporamos también los eventos ya cargados

dataframes["usage_events_stream"] = events


print("Fuentes cargadas para exploración:")

for nombre in dataframes.keys():
    print("-", nombre)

## 3.2 Grano, clave y modalidad esperada

Antes de analizar la calidad se documenta qué representa una fila de cada fuente.

El **grano** define el nivel de detalle de una fuente. Conocerlo resulta fundamental para realizar joins y agregaciones correctamente y evitar duplicar información.

También se identifica una clave candidata para cada fuente y la modalidad de procesamiento inicialmente esperada.

Cuando la consigna no establece una periodicidad exacta para una fuente batch, se deja explícitamente como una decisión pendiente en lugar de asumir una frecuencia no especificada.

In [ ]:
catalogo_fuentes = [
    (
        "customers_orgs",
        "CSV",
        "Una organización",
        "org_id",
        "Batch - maestro; periodicidad a definir",
        "signup_date"
    ),
    (
        "users",
        "CSV",
        "Un usuario",
        "user_id",
        "Batch - maestro; periodicidad a definir",
        "created_at / last_login"
    ),
    (
        "resources",
        "CSV",
        "Un recurso cloud",
        "resource_id",
        "Batch - maestro; periodicidad a definir",
        "created_at"
    ),
    (
        "support_tickets",
        "CSV",
        "Un ticket de soporte",
        "ticket_id",
        "Batch inicial; periodicidad a definir",
        "created_at / resolved_at"
    ),
    (
        "marketing_touches",
        "CSV",
        "Una interacción de marketing",
        "touch_id",
        "Batch inicial; periodicidad a definir",
        "timestamp"
    ),
    (
        "nps_surveys",
        "CSV",
        "Una encuesta de una organización en una fecha",
        "org_id + survey_date",
        "Batch inicial; periodicidad a definir",
        "survey_date"
    ),
    (
        "billing_monthly",
        "CSV",
        "Una factura mensual",
        "invoice_id",
        "Batch mensual",
        "month"
    ),
    (
        "usage_events_stream",
        "JSONL",
        "Un evento de utilización",
        "event_id",
        "Streaming / micro-lotes simulados",
        "timestamp"
    )
]

catalogo_df = spark.createDataFrame(
    catalogo_fuentes,
    [
        "fuente",
        "formato",
        "grano",
        "clave_candidata",
        "modalidad",
        "campo_temporal"
    ]
)

catalogo_df.show(truncate=False)

## 3.3 Estructura y columnas observadas

Se inspeccionarán las columnas y el esquema con el que Spark representa inicialmente cada fuente.

Los CSV fueron leídos deliberadamente sin `inferSchema`, por lo que sus columnas permanecerán inicialmente como texto. Esto no significa que todos los campos sean conceptualmente de tipo string.

Por ejemplo, fechas, booleanos e importes deberán recibir posteriormente una tipificación explícita.

Esta separación entre dato crudo y tipo objetivo evita depender de inferencias automáticas durante el pipeline.

In [ ]:
for nombre, df in dataframes.items():

    print("\n" + "=" * 80)
    print("FUENTE:", nombre)
    print("=" * 80)

    print("Cantidad de registros:", df.count())
    print("Cantidad de columnas:", len(df.columns))

    print("\nColumnas:")
    print(df.columns)

    print("\nEsquema observado:")
    df.printSchema()

## 3.4 Perfil de valores nulos

Se analizará la presencia de valores nulos o vacíos en todas las fuentes.

La existencia de un valor nulo no implica automáticamente un error.

Dependiendo del significado del campo, puede representar:

- información faltante;
- información todavía no disponible;
- un atributo que no corresponde al registro;
- una condición válida del negocio.

Por este motivo, esta etapa únicamente identifica los nulos. La decisión sobre cómo tratarlos se realizará posteriormente según la semántica de cada columna.

In [ ]:
def obtener_perfil_nulos(nombre_fuente, df):

    total = df.count()

    expresiones = []

    for columna in df.columns:

        valor_como_texto = F.trim(
            F.col(columna).cast("string")
        )

        expresiones.append(
            F.sum(
                F.when(
                    F.col(columna).isNull() |
                    (valor_como_texto == ""),
                    1
                ).otherwise(0)
            ).alias(columna)
        )

    resultado = df.agg(*expresiones).first().asDict()

    filas = []

    for columna, cantidad_nulos in resultado.items():

        if cantidad_nulos > 0:

            porcentaje = round(
                (cantidad_nulos / total) * 100,
                2
            )

            filas.append(
                (
                    nombre_fuente,
                    columna,
                    total,
                    cantidad_nulos,
                    porcentaje
                )
            )

    return filas


perfil_nulos = []

for nombre, df in dataframes.items():

    perfil_nulos.extend(
        obtener_perfil_nulos(nombre, df)
    )


perfil_nulos_df = spark.createDataFrame(
    perfil_nulos,
    [
        "fuente",
        "columna",
        "registros_totales",
        "nulos",
        "porcentaje_nulos"
    ]
)

perfil_nulos_df.orderBy(
    "fuente",
    F.desc("porcentaje_nulos")
).show(100, truncate=False)

## 3.5 Análisis de duplicados según claves candidatas

La existencia de filas completamente iguales no es la única forma de detectar duplicados.

Cada fuente posee un grano y una clave candidata. Por lo tanto, se verificará si existen múltiples registros con la misma clave lógica.

Esta comprobación es relevante para evitar duplicar entidades, tickets, facturas o eventos durante el procesamiento.

In [ ]:
claves_candidatas = {
    "customers_orgs": ["org_id"],
    "users": ["user_id"],
    "resources": ["resource_id"],
    "support_tickets": ["ticket_id"],
    "marketing_touches": ["touch_id"],
    "nps_surveys": ["org_id", "survey_date"],
    "billing_monthly": ["invoice_id"],
    "usage_events_stream": ["event_id"]
}


resultado_duplicados = []

for nombre, columnas_clave in claves_candidatas.items():

    df = dataframes[nombre]

    grupos_duplicados = (
        df
        .groupBy(*columnas_clave)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    resultado_duplicados.append(
        (
            nombre,
            " + ".join(columnas_clave),
            grupos_duplicados
        )
    )


duplicados_df = spark.createDataFrame(
    resultado_duplicados,
    [
        "fuente",
        "clave_evaluada",
        "grupos_duplicados"
    ]
)

duplicados_df.show(truncate=False)

### Validación adicional de facturación

Además de verificar que `invoice_id` sea único, se comprobará que no exista más de una factura para la misma combinación organización-mes dentro del dataset provisto.

Esta validación permite confirmar el grano observado de la fuente mensual.

In [ ]:
duplicados_org_mes = (
    dataframes["billing_monthly"]
    .groupBy("org_id", "month")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Combinaciones org_id + month duplicadas:",
    duplicados_org_mes
)

## 3.6 Integridad referencial entre fuentes

Muchas fuentes contienen `org_id` para indicar a qué organización pertenece cada registro.

Se comprobará si todos los `org_id` utilizados por las fuentes relacionadas existen realmente en el maestro `customers_orgs`.

También se verificará que los `resource_id` utilizados en los eventos de usage existan en el maestro de recursos.

Para detectar referencias inexistentes se utilizarán joins de Spark.

In [ ]:
organizaciones = (
    dataframes["customers_orgs"]
    .select("org_id")
    .distinct()
)


fuentes_con_org = [
    "users",
    "resources",
    "support_tickets",
    "marketing_touches",
    "nps_surveys",
    "billing_monthly",
    "usage_events_stream"
]


resultado_integridad_org = []

for nombre in fuentes_con_org:

    ids_huerfanos = (
        dataframes[nombre]
        .select("org_id")
        .distinct()
        .join(
            organizaciones,
            on="org_id",
            how="left_anti"
        )
        .count()
    )

    resultado_integridad_org.append(
        (
            nombre,
            ids_huerfanos
        )
    )


integridad_org_df = spark.createDataFrame(
    resultado_integridad_org,
    [
        "fuente",
        "org_id_sin_maestro"
    ]
)

integridad_org_df.show(truncate=False)

In [ ]:
recursos = (
    dataframes["resources"]
    .select("resource_id")
    .distinct()
)


resources_huerfanos = (
    dataframes["usage_events_stream"]
    .select("resource_id")
    .distinct()
    .join(
        recursos,
        on="resource_id",
        how="left_anti"
    )
    .count()
)


print(
    "resource_id de eventos sin recurso maestro:",
    resources_huerfanos
)

## 3.7 Cobertura temporal de las fuentes

Además de conocer la cantidad de registros, resulta útil identificar el período temporal cubierto por las fuentes.

Esto permite comprender si los datasets poseen ventanas temporales comparables y ayuda posteriormente a diseñar procesos incrementales, particiones y agregaciones.

La cobertura temporal observada no debe confundirse con la frecuencia real de ingesta.

Por ejemplo, conocer que una fuente contiene cuatro meses de información no permite afirmar que sea ingerida diariamente.

In [ ]:
campos_temporales = [
    ("customers_orgs", "signup_date"),
    ("users", "created_at"),
    ("users", "last_login"),
    ("resources", "created_at"),
    ("support_tickets", "created_at"),
    ("support_tickets", "resolved_at"),
    ("marketing_touches", "timestamp"),
    ("nps_surveys", "survey_date"),
    ("billing_monthly", "month"),
    ("usage_events_stream", "timestamp")
]


cobertura_temporal = []


for nombre, columna in campos_temporales:

    df = dataframes[nombre]

    df_temporal = df.select(
        F.to_timestamp(F.col(columna)).alias("fecha")
    )

    resultado = (
        df_temporal
        .agg(
            F.min("fecha").alias("desde"),
            F.max("fecha").alias("hasta"),
            F.sum(
                F.when(
                    F.col("fecha").isNull(),
                    1
                ).otherwise(0)
            ).alias("fechas_nulas")
        )
        .first()
    )

    cobertura_temporal.append(
        (
            nombre,
            columna,
            resultado["desde"],
            resultado["hasta"],
            resultado["fechas_nulas"]
        )
    )


cobertura_temporal_df = spark.createDataFrame(
    cobertura_temporal,
    [
        "fuente",
        "campo",
        "desde",
        "hasta",
        "valores_nulos"
    ]
)

cobertura_temporal_df.show(
    100,
    truncate=False
)

## 3.8 Trazabilidad del archivo de origen

Además de conocer el contenido de cada registro, un pipeline de datos debe poder identificar de qué archivo provino.

Esta información resulta especialmente relevante en la fuente de eventos, debido a que se encuentra fragmentada en múltiples archivos.

Spark permite recuperar el archivo de origen de cada registro mediante `input_file_name()`.

En la futura capa Bronze esta información podrá conservarse mediante una columna técnica como `source_file`, acompañada por un timestamp de ingesta.

In [ ]:
events_con_origen = (
    spark.read
    .json(str(stream_path / "*.jsonl"))
    .withColumn(
        "source_file",
        F.input_file_name()
    )
)


cantidad_archivos_detectados = (
    events_con_origen
    .select("source_file")
    .distinct()
    .count()
)


print(
    "Archivos de origen detectados por Spark:",
    cantidad_archivos_detectados
)


events_con_origen.select(
    "event_id",
    "timestamp",
    "service",
    "source_file"
).show(
    5,
    truncate=False
)

## 3.9 Perfil inicial y riesgos identificados

La exploración de las fuentes permite establecer una primera caracterización del dataset.

### customers_orgs

**Grano:** una organización.  
**Clave candidata:** `org_id`.  
**Modalidad esperada:** batch.  
**Observaciones:** contiene información descriptiva de las organizaciones y algunos valores nulos en `nps_score`. La consigna anticipa además valores ruidosos, por lo que sus dominios deberán validarse antes de utilizarse como dimensión conformada.

---

### users

**Grano:** un usuario.  
**Clave candidata:** `user_id`.  
**Relación principal:** `org_id`.  
**Modalidad esperada:** batch.  
**Observaciones:** existen valores nulos en `last_login`. Estos valores no deben considerarse automáticamente errores, ya que será necesario determinar si representan usuarios que todavía no iniciaron sesión.

---

### resources

**Grano:** un recurso cloud.  
**Clave candidata:** `resource_id`.  
**Relación principal:** `org_id`.  
**Modalidad esperada:** batch.  
**Observaciones:** `tags_json` presenta valores nulos. La fuente contiene diferentes servicios y regiones que posteriormente deberán conformarse con los eventos.

---

### support_tickets

**Grano:** un ticket.  
**Clave candidata:** `ticket_id`.  
**Relación principal:** `org_id`.  
**Modalidad esperada:** batch.  
**Observaciones:** existen valores nulos en `resolved_at` y `csat`. Será necesario distinguir tickets abiertos de datos efectivamente faltantes antes de definir reglas de calidad.

---

### marketing_touches

**Grano:** una interacción de marketing.  
**Clave candidata:** `touch_id`.  
**Relación principal:** `org_id`.  
**Modalidad esperada:** batch.  
**Observaciones:** no se detectaron nulos relevantes durante la exploración inicial.

---

### nps_surveys

**Grano:** una encuesta correspondiente a una organización y una fecha.  
**Clave candidata observada:** `org_id + survey_date`.  
**Modalidad esperada:** batch.  
**Observaciones:** existen valores nulos en `nps_score` y `comment`. Será necesario establecer las reglas de dominio antes de determinar qué valores pueden considerarse inválidos.

---

### billing_monthly

**Grano:** una factura mensual.  
**Clave candidata:** `invoice_id`.  
**Relación principal:** `org_id`.  
**Modalidad esperada:** batch mensual.  
**Observaciones:** contiene tres meses de información y presenta valores nulos en `credits`. Las monedas y tasas de conversión deberán considerarse posteriormente al calcular métricas de revenue normalizadas.

---

### usage_events_stream

**Grano:** un evento de utilización.  
**Clave candidata:** `event_id`.  
**Relaciones principales:** `org_id` y `resource_id`.  
**Modalidad esperada:** streaming mediante micro-lotes.  
**Observaciones:** constituye la fuente con mayor cantidad de registros y presenta evolución de esquema, tipos ambiguos, valores nulos y costos negativos. Será la principal fuente que requerirá compatibilización, deduplicación, control de calidad y procesamiento incremental.

---

## Resultados generales

La exploración inicial permitió comprobar que:

- las ocho fuentes previstas están disponibles;
- las claves candidatas evaluadas no presentan duplicados en el dataset actual;
- los `org_id` utilizados por las fuentes relacionadas poseen correspondencia con el maestro de organizaciones;
- los `resource_id` utilizados por los eventos poseen correspondencia con el maestro de recursos;
- existen valores nulos cuya interpretación depende del contexto de cada campo;
- los eventos presentan dos versiones de esquema;
- los archivos de eventos permiten conservar trazabilidad hacia el archivo de origen;
- el dataset combina fuentes batch con una fuente diseñada para simular procesamiento streaming.

Estos resultados constituyen la base para definir posteriormente los esquemas explícitos, reglas de calidad, zonas del Data Lake y flujos de procesamiento.

## 3.10 Riesgos iniciales derivados del perfil de datos

| Riesgo observado | Fuente | Consecuencia potencial | Tratamiento a diseñar |
|---|---|---|---|
| Valores nulos | Varias fuentes | Métricas incompletas o interpretaciones incorrectas | Reglas específicas según la semántica del campo |
| Tipos ambiguos en `value` | Usage events | Errores en cálculos y agregaciones | Tipificación explícita y casting controlado |
| Evolución schema v1/v2 | Usage events | Columnas no disponibles para todo el histórico | Compatibilización de versiones |
| Costos negativos | Usage events | Distorsión de métricas de costos | Validación y flag de anomalía |
| Valores potencialmente atípicos | Fuentes numéricas | Distorsión de agregaciones | Perfil estadístico y reglas justificadas |
| Late data | Usage events | Métricas temporales incompletas | Watermark en la etapa de streaming |
| Posibles duplicados futuros | Usage events | Doble contabilización | Deduplicación mediante `event_id` |
| Cambios o inconsistencias de dominios | Servicios, regiones y categorías | Joins o agrupaciones inconsistentes | Normalización en Silver |
| Pérdida de trazabilidad | Todas las fuentes | Dificultad para auditar errores | `source_file`, `ingest_ts` y metadatos técnicos |

Los riesgos anteriores representan problemas que deberán resolverse progresivamente en Bronze y Silver. En esta primera evaluación se identifican y documentan; su implementación completa corresponde a las siguientes etapas del proyecto.

# 4 - Arquitectura de alto nivel

A partir del análisis del problema y del perfil inicial de las fuentes se propone una arquitectura de datos que permita integrar información procesada mediante batch con eventos que requieren procesamiento incremental.

La arquitectura sigue la cadena lógica:

**Fuentes → Ingesta → Data Lake → Procesamiento → Serving → Consumo**

y contempla de forma transversal:

- calidad;
- metadatos;
- trazabilidad y linaje;
- seguridad;
- observabilidad.

En esta primera evaluación se presenta el diseño conceptual de la arquitectura. La implementación técnica progresiva de sus componentes se realizará en las siguientes etapas del proyecto.

## 4.1 Fuentes de datos

La arquitectura recibe ocho fuentes lógicas actualmente disponibles en Landing.

### Fuentes destinadas principalmente a procesamiento batch

- `customers_orgs.csv`
- `users.csv`
- `resources.csv`
- `support_tickets.csv`
- `marketing_touches.csv`
- `nps_surveys.csv`
- `billing_monthly.csv`

Estas fuentes corresponden principalmente a maestros, información transaccional y datos que no requieren procesamiento evento a evento.

### Fuente destinada a procesamiento streaming

- `usage_events_stream/*.jsonl`

Esta fuente contiene eventos de utilización fragmentados en micro-lotes simulados y será procesada posteriormente mediante Structured Streaming.

Los archivos originales serán preservados sin modificaciones dentro de Landing.

## 4.2 Vista general de la arquitectura

La solución se organiza de la siguiente manera:

```text
                           FUENTES
                              │
             ┌────────────────┴─────────────────┐
             │                                  │
          CSV batch                       JSONL eventos
             │                                  │
             ▼                                  ▼
       ┌─────────────────────────────────────────────┐
       │                  LANDING                    │
       │             Datos originales                │
       │           crudos e inmutables               │
       └─────────────────────────────────────────────┘
             │                                  │
             │ Batch                            │ Streaming
             ▼                                  ▼
       PySpark DataFrames              Structured Streaming
             │                                  │
             └────────────────┬─────────────────┘
                              ▼
                    ┌──────────────────┐
                    │      BRONZE      │
                    │   Raw estándar   │
                    │     Parquet      │
                    └────────┬─────────┘
                             │
                           PySpark
                             │
                             ▼
                    ┌──────────────────┐
                    │      SILVER      │
                    │ Datos conformados│
                    │     Parquet      │
                    └────────┬─────────┘
                             │
                           PySpark
                             │
                             ▼
                    ┌──────────────────┐
                    │       GOLD       │
                    │ Marts analíticos │
                    │     Parquet      │
                    └────────┬─────────┘
                             │
                             ▼
                  Cassandra / AstraDB
                       Serving layer
                             │
            ┌────────────────┼─────────────────┐
            │                │                 │
            ▼                ▼                 ▼
         FinOps           Soporte       Producto / Usage


       CAPACIDADES TRANSVERSALES

       Calidad · Metadatos · Linaje · Seguridad
                   · Observabilidad
```

Este esquema representa la arquitectura lógica de alto nivel. Para el documento final se utilizará una versión gráfica del mismo diseño, manteniendo los mismos componentes y flujos.

## 4.3 Responsabilidad de los componentes

### Fuentes

Representan los sistemas o archivos que originan la información utilizada por el proyecto.

Los datos presentan diferentes formatos, dominios, frecuencias y características de calidad.

---

### Ingesta

La ingesta tendrá dos caminos principales.

**Batch**

Se utilizará PySpark para leer las fuentes que pueden procesarse periódicamente, como maestros, facturación, soporte y encuestas.

**Streaming**

Los eventos de `usage_events_stream` serán procesados mediante Structured Streaming, utilizando una lógica incremental sobre los archivos JSONL disponibles.

Ambos caminos convergen en el Data Lake.

---

### Landing

Contiene los archivos originales recibidos.

Características principales:

- conserva el formato de origen;
- no modifica los registros;
- permite reproducir el procesamiento desde la fuente original;
- constituye el punto inicial de trazabilidad.

---

### Bronze

Representa los datos crudos estandarizados técnicamente.

Se mantendrá el mismo grano de las fuentes y se incorporarán progresivamente:

- esquemas explícitos;
- tipos definidos;
- `ingest_ts`;
- `source_file`;
- deduplicación cuando corresponda.

El formato previsto para esta zona es Parquet.

---

### Silver

Contiene datos limpios y conformados.

En esta zona se realizarán posteriormente tareas como:

- normalización de tipos;
- tratamiento de nulos;
- normalización de regiones y servicios;
- compatibilización de schema v1 y v2;
- tratamiento o identificación de outliers;
- joins con dimensiones;
- reglas de calidad y negocio.

El resultado será información consistente y reutilizable por diferentes análisis.

---

### Gold

Contiene información preparada para casos de uso concretos del negocio.

En lugar de conservar simplemente las fuentes originales, se crearán datasets con granos definidos para responder preguntas analíticas de FinOps, Soporte y Producto/Usage.

Ejemplos previstos para etapas posteriores:

- costos y requests diarios por organización y servicio;
- revenue mensual por organización;
- anomalías de costos;
- indicadores diarios de soporte;
- utilización de GenAI.

---

### Serving

Los datos preparados en Gold serán publicados posteriormente en Cassandra/AstraDB.

El modelo de almacenamiento deberá diseñarse orientado a las consultas que necesiten realizar los consumidores.

---

### Consumo

La información publicada será utilizada principalmente por:

- FinOps;
- Soporte;
- Producto / Usage;

y posteriormente podrá ser consumida desde herramientas de visualización o componentes analíticos.

## 4.4 Capacidades transversales

Además del flujo principal de datos, la arquitectura deberá contemplar capacidades que atraviesan todas las etapas.

### Calidad

Se establecerán reglas verificables para detectar datos inválidos, inconsistentes o anómalos.

Ejemplos ya identificados durante la exploración:

- tipos ambiguos;
- valores nulos;
- costos negativos;
- evolución del esquema;
- posibles duplicados futuros.

Los registros que no puedan ser promovidos deberán poder ser identificados y, cuando corresponda, separados para su revisión.

---

### Metadatos y trazabilidad

Cada etapa deberá permitir conocer:

- fuente de origen;
- archivo de origen;
- momento de ingesta;
- transformaciones realizadas;
- zona actual del dato.

Las columnas técnicas como `source_file` e `ingest_ts` contribuirán a esta trazabilidad.

---

### Linaje

Se documentará cómo una fuente de Landing se transforma progresivamente hasta producir información Silver, Gold y posteriormente datos publicados en serving.

Esto permitirá relacionar una métrica final con los datos y procesos que la originaron.

---

### Seguridad

Las credenciales y configuraciones sensibles no deberán almacenarse directamente dentro de notebooks ni del repositorio.

Los accesos a los componentes de serving deberán mantenerse separados de la lógica del procesamiento.

---

### Observabilidad

El pipeline deberá generar evidencia que permita conocer su ejecución.

Entre las métricas previstas se encuentran:

- registros leídos;
- registros procesados;
- registros inválidos;
- duplicados;
- archivos procesados;
- errores;
- tiempos de ejecución.

Estas evidencias facilitarán posteriormente la validación y reproducción del pipeline.

## 4.5 Relación entre problemas observados y arquitectura

El perfil inicial realizado sobre las fuentes permite justificar algunas de las responsabilidades asignadas a la arquitectura.

| Evidencia observada | Necesidad arquitectónica |
|---|---|
| Archivos originales CSV y JSONL | Zona Landing inmutable |
| Diferentes tipos y formatos | Ingesta y tipificación controlada |
| 120 fragmentos de eventos | Flujo preparado para Structured Streaming |
| `value` puede llegar como texto | Normalización y casting en Silver |
| Dos versiones de esquema | Compatibilización v1/v2 en Silver |
| Valores nulos | Reglas de calidad según contexto |
| Costos negativos | Validaciones y detección de anomalías |
| Relaciones entre organizaciones, recursos y eventos | Joins de enriquecimiento en Silver |
| Necesidad de costos diarios y otras métricas | Marts específicos en Gold |
| Necesidad de consultas posteriores | Serving en Cassandra/AstraDB |
| Necesidad de auditar el origen | `source_file`, `ingest_ts` y metadatos |

De esta manera, los componentes de la arquitectura responden a características comprobadas del dataset o a requisitos explícitos del caso y no se incorporan únicamente como decisiones tecnológicas.

## 4.6 Alcance de implementación por etapa

La arquitectura presentada corresponde al diseño objetivo del proyecto completo, pero su implementación será progresiva.

### Primera evaluación

Se desarrolla:

- comprensión del problema;
- perfil de fuentes;
- diseño arquitectónico;
- diseño del Data Lake;
- flujos batch y streaming;
- decisiones iniciales;
- evidencia de exploración.

### Segunda evaluación

Se implementará un flujo mínimo ejecutable que incluya:

- ingesta batch;
- ingesta streaming;
- Bronze;
- Silver;
- un mart Gold;
- reglas de calidad;
- serving inicial en Cassandra/AstraDB.

### Evaluación final

Se completará el flujo end-to-end y los dominios analíticos requeridos, incorporando mayor nivel de calidad, gobierno, pruebas, documentación y reproducibilidad.

## 4.7 Conclusión de la arquitectura v1

La arquitectura propuesta separa claramente las responsabilidades de ingestión, almacenamiento, procesamiento y consumo.

Las principales decisiones iniciales son:

1. preservar las fuentes originales en Landing;
2. diferenciar la ingesta batch de los eventos destinados a streaming;
3. utilizar PySpark como motor principal de procesamiento;
4. utilizar Parquet como formato intermedio del Data Lake;
5. separar progresivamente los datos en Bronze, Silver y Gold;
6. publicar posteriormente los marts analíticos mediante Cassandra/AstraDB;
7. mantener calidad, trazabilidad, seguridad, metadatos y observabilidad como capacidades transversales.

La arquitectura será refinada durante las siguientes etapas a medida que se implementen y validen sus componentes.

# 5 - Selección y justificación del patrón arquitectónico

La solución debe combinar dos necesidades diferentes:

1. procesamiento batch para fuentes maestras, facturación y otras fuentes que no requieren procesamiento evento a evento;
2. procesamiento streaming para los eventos de utilización que requieren actualización cercana al tiempo real.

A partir de estas características se evaluarán los patrones arquitectónicos propuestos para el proyecto y se seleccionará el que mejor represente el caso.

## 5.1 Alternativas consideradas

### Procesamiento únicamente batch

Una arquitectura exclusivamente batch permitiría procesar periódicamente las fuentes disponibles.

Sin embargo, no resulta suficiente para el caso porque `usage_events_stream` debe contemplar procesamiento near real-time mediante Structured Streaming.

Por este motivo, una solución exclusivamente batch no cumple el alcance completo del proyecto.

---

### Arquitectura Lambda

La arquitectura Lambda combina dos caminos de procesamiento:

- **batch**, para datos que pueden procesarse periódicamente;
- **streaming**, para información que requiere procesamiento incremental o cercano al tiempo real.

En este proyecto permitiría utilizar:

**Batch**

- clientes;
- usuarios;
- recursos;
- tickets;
- marketing;
- NPS;
- facturación.

**Streaming**

- eventos de utilización de servicios cloud.

Ambos caminos pueden converger posteriormente en capas conformadas y productos analíticos comunes.

---

### Arquitectura Kappa

Una arquitectura Kappa prioriza el tratamiento de las fuentes como streams y utiliza mecanismos de re-stream o backfill cuando es necesario reprocesar información histórica.

Este enfoque no representa de manera natural la estructura actual del caso, debido a que varias de las fuentes provistas son maestros o archivos periódicos que la propia consigna requiere procesar mediante batch.

---

### Arquitectura híbrida

Una arquitectura híbrida permite combinar diferentes patrones cuando existen necesidades que no son resueltas adecuadamente mediante una única estrategia.

En este proyecto no se observa inicialmente una necesidad adicional que justifique incorporar mayor complejidad respecto de la combinación batch + streaming ya contemplada por Lambda.

## 5.2 Patrón seleccionado: Lambda

Se selecciona una **arquitectura Lambda**, debido a que representa de manera directa las dos necesidades de procesamiento identificadas en el caso.

### Camino batch

Las fuentes maestras y periódicas serán procesadas mediante PySpark en ejecuciones batch.

Este camino incluirá principalmente:

- `customers_orgs.csv`;
- `users.csv`;
- `resources.csv`;
- `support_tickets.csv`;
- `marketing_touches.csv`;
- `nps_surveys.csv`;
- `billing_monthly.csv`.

Estas fuentes no requieren procesamiento individual de cada registro en el momento en que es generado.

### Camino streaming

La fuente:

`usage_events_stream/*.jsonl`

será procesada mediante Structured Streaming.

Este camino permitirá posteriormente incorporar:

- esquema explícito;
- procesamiento incremental;
- watermark;
- deduplicación mediante `event_id`;
- manejo de eventos tardíos;
- checkpointing.

Los datos provenientes de ambos caminos serán preparados progresivamente para alimentar las capas Silver y Gold.

## 5.3 Comparación de alternativas

| Criterio | Batch solamente | Lambda | Kappa | Híbrido |
|---|---|---|---|---|
| Procesamiento de maestros | Sí | Sí | Posible, pero no natural para el caso | Sí |
| Facturación periódica | Sí | Sí | Requeriría tratarla como stream o reprocesarla | Sí |
| Usage events near real-time | No | Sí | Sí | Sí |
| Diferencia claramente batch y streaming | No | Sí | No como caminos separados | Sí |
| Se ajusta directamente a las fuentes provistas | Parcialmente | Sí | Parcialmente | Sí |
| Complejidad adicional necesaria para este caso | Baja, pero insuficiente | Adecuada | Mayor para las fuentes batch | No justificada inicialmente |

La arquitectura Lambda permite cumplir los requisitos del caso sin transformar artificialmente las fuentes batch en streams ni perder la capacidad de procesar eventos de utilización en forma incremental.

## 5.4 Aplicación del patrón al proyecto

El patrón seleccionado puede representarse conceptualmente de la siguiente manera:

```text
                           LANDING
                              │
              ┌───────────────┴───────────────┐
              │                               │
              ▼                               ▼
        FUENTES BATCH                  USAGE EVENTS
              │                               │
              ▼                               ▼
           PySpark                  Structured Streaming
              │                               │
              ▼                               ▼
          BRONZE                           BRONZE
              │                               │
              └───────────────┬───────────────┘
                              ▼
                           SILVER
                              │
                              ▼
                            GOLD
                              │
                              ▼
                    Cassandra / AstraDB
```

## 5.5 Decisión arquitectónica

Se adopta el patrón **Lambda** como arquitectura de referencia para el proyecto.

La decisión se fundamenta en:

1. la existencia de fuentes maestras y periódicas adecuadas para procesamiento batch;
2. la existencia de eventos de utilización que requieren procesamiento streaming;
3. la necesidad explícita de distinguir ambos tipos de procesamiento;
4. la posibilidad de utilizar PySpark y Structured Streaming dentro de un mismo ecosistema;
5. la posibilidad de converger posteriormente en capas Silver y Gold comunes;
6. la menor complejidad respecto de tratar artificialmente todas las fuentes como streams.

Esta decisión podrá revisarse durante las siguientes etapas si la implementación o nuevos requisitos demuestran la necesidad de modificar el patrón.

# 6 - Mapeo de requisitos a componentes

Una arquitectura debe poder relacionarse directamente con los problemas y requisitos que intenta resolver.

Por este motivo, se establece una matriz de trazabilidad entre:

- necesidad o requisito;
- evidencia o característica del caso;
- componente arquitectónico propuesto;
- decisión técnica asociada.

El objetivo es evitar incorporar componentes tecnológicos sin una necesidad concreta y facilitar posteriormente la validación de que la implementación responde al diseño.

## 6.1 Matriz requisito → componente

| Necesidad / requisito | Evidencia o motivo | Componente propuesto | Responsabilidad |
|---|---|---|---|
| Conservar los datos originales | Las fuentes deben permanecer inmutables | Landing | Preservar CSV y JSONL sin modificaciones |
| Procesar maestros y fuentes periódicas | Clientes, usuarios, recursos, facturación, tickets, marketing y NPS se reciben como archivos | PySpark Batch | Realizar lecturas y procesamiento periódico |
| Procesar eventos near real-time | `usage_events_stream` representa micro-lotes de eventos | Structured Streaming | Procesamiento incremental de eventos |
| Mantener un formato intermedio eficiente | Las siguientes etapas requieren lectura y transformación repetida | Parquet | Almacenamiento columnar para Bronze, Silver y Gold |
| Estandarizar técnicamente las fuentes | Los datos llegan con distintos formatos y tipos | Bronze | Aplicar esquemas definidos y agregar columnas técnicas |
| Mantener trazabilidad | Es necesario identificar archivo y momento de ingreso | `source_file` + `ingest_ts` | Registrar procedencia e ingesta |
| Evitar duplicación de eventos | Aunque actualmente no se detectaron duplicados, pueden aparecer durante re-ejecuciones o streaming | `event_id` + deduplicación | Mantener idempotencia |
| Resolver tipos ambiguos | Se encontraron valores de `value` como número y como texto | Silver | Casting y normalización controlada |
| Compatibilizar evolución de esquema | Existen schema v1 y v2 | Silver | Unificar estructuras y campos disponibles |
| Tratar valores nulos según contexto | Se observaron nulos en varias fuentes | Silver + reglas de calidad | Diferenciar nulos válidos de problemas de calidad |
| Controlar costos negativos y anomalías | Se detectaron costos negativos y el caso contempla outliers | Calidad / Silver | Validar, identificar y tratar anomalías |
| Relacionar eventos con organizaciones y recursos | Los eventos poseen `org_id` y `resource_id` | Joins en Silver | Enriquecer eventos mediante dimensiones |
| Calcular métricas de negocio | FinOps, Soporte y Producto requieren información agregada | Gold | Construir marts con granos analíticos definidos |
| Consultar los resultados de forma eficiente | Los consumidores necesitan acceder a información preparada | Cassandra / AstraDB | Serving orientado a consultas |
| Auditar el procesamiento | La solución debe ser reproducible y verificable | Metadatos + linaje + observabilidad | Registrar ejecución, origen y transformaciones |
| Proteger configuración sensible | El repositorio no debe contener credenciales | Seguridad / configuración externa | Separar secretos de código y documentación |

## 6.2 Relación entre las 5V y las decisiones de arquitectura

El análisis de las 5V realizado previamente también permite justificar decisiones concretas de diseño.

| Dimensión | Evidencia / característica | Decisión arquitectónica |
|---|---|---|
| **Volumen** | Los eventos constituyen la fuente de mayor crecimiento y el escenario representa generación continua de información | Utilizar Apache Spark y almacenamiento Parquet, permitiendo escalar el procesamiento y organizar los datos por capas |
| **Velocidad** | Los eventos se encuentran fragmentados en micro-lotes y el caso requiere procesamiento near real-time | Utilizar Structured Streaming para `usage_events_stream` |
| **Variedad** | Existen CSV, JSONL, múltiples dominios, servicios y métricas | Separar Landing, Bronze y Silver para preservar, estandarizar y conformar las fuentes |
| **Veracidad** | Se observaron nulos, tipos ambiguos, costos negativos y dos versiones de esquema | Incorporar reglas de calidad, compatibilización de esquema y normalización en Silver |
| **Valor** | FinOps, Soporte y Producto necesitan responder preguntas analíticas diferentes | Construir marts específicos en Gold y publicarlos posteriormente mediante Cassandra/AstraDB |

La relación anterior muestra que las 5V no se utilizan únicamente como una justificación teórica de Big Data, sino también como criterios que influyen en las decisiones arquitectónicas.

## 6.3 Relación entre usuarios y productos analíticos

| Dominio | Necesidad principal | Salida prevista |
|---|---|---|
| FinOps | Analizar costos, consumo, revenue y anomalías | Marts Gold de costos y facturación |
| Soporte | Analizar volumen de tickets, severidad, SLA y satisfacción | Mart Gold de indicadores de soporte |
| Producto / Usage | Analizar utilización, requests, GenAI y carbono | Marts Gold de utilización y GenAI |

Los productos Gold se diseñarán con un grano específico para las consultas que cada dominio necesita responder.

## 6.4 Conclusión de trazabilidad

La matriz permite verificar que los principales componentes de la arquitectura responden a requisitos concretos del proyecto.

Las decisiones centrales pueden resumirse de la siguiente manera:

- Landing responde a la necesidad de preservar los datos originales.
- Bronze responde a la necesidad de estandarizar técnicamente la ingesta.
- Silver responde a los problemas de calidad, normalización, joins y evolución de esquema.
- Gold responde a las necesidades analíticas de los usuarios.
- PySpark Batch responde al procesamiento de maestros y fuentes periódicas.
- Structured Streaming responde a la necesidad de procesar eventos incrementales.
- Parquet constituye el formato intermedio del Data Lake.
- Cassandra/AstraDB constituye la capa de serving prevista.
- calidad, metadatos, linaje, seguridad y observabilidad atraviesan todo el proceso.

Esta trazabilidad será utilizada posteriormente para comprobar que la implementación técnica conserve coherencia con la arquitectura diseñada.

# 7 - Diseño del Data Lake

El Data Lake se organizará en cuatro zonas principales:

1. **Landing:** preservación de los archivos originales.
2. **Bronze:** datos crudos estandarizados técnicamente.
3. **Silver:** datos limpios y conformados.
4. **Gold:** productos de datos preparados para consumo analítico.

El diseño debe permitir mantener trazabilidad desde los archivos originales hasta los productos analíticos finales y soportar tanto el procesamiento batch como el procesamiento streaming.

Para cada zona se definirán:

- responsabilidad;
- formato;
- organización;
- particionamiento;
- metadatos;
- reglas de promoción;
- criterio de retención.

## 7.1 Organización general del Data Lake

Se propone la siguiente estructura lógica:

```text
datalake/
│
├── landing/
│   ├── customers_orgs/
│   ├── users/
│   ├── resources/
│   ├── support_tickets/
│   ├── marketing_touches/
│   ├── nps_surveys/
│   ├── billing_monthly/
│   └── usage_events_stream/
│
├── bronze/
│   ├── customers_orgs/
│   ├── users/
│   ├── resources/
│   ├── support_tickets/
│   ├── marketing_touches/
│   ├── nps_surveys/
│   ├── billing_monthly/
│   └── usage_events/
│
├── silver/
│   ├── customers_orgs/
│   ├── users/
│   ├── resources/
│   ├── support_tickets/
│   ├── billing_monthly/
│   └── usage_events/
│
├── gold/
│   ├── finops/
│   ├── support/
│   └── product/
│
└── quarantine/
    └── registros_invalidos/
```

## 7.2 Responsabilidad y formato de cada zona

| Zona | Responsabilidad | Formato principal | Transformación |
|---|---|---|---|
| **Landing** | Preservar los datos exactamente como fueron recibidos | CSV y JSONL originales | Ninguna |
| **Bronze** | Estandarizar técnicamente la ingesta manteniendo el mismo grano | Parquet | Tipificación explícita, metadatos técnicos y deduplicación cuando corresponda |
| **Silver** | Limpiar, normalizar, conformar y enriquecer los datos | Parquet | Calidad, joins, tratamiento de nulos/outliers y compatibilidad de esquema |
| **Gold** | Preparar productos de datos orientados a consultas de negocio | Parquet | Agregaciones y métricas con grano analítico definido |
| **Quarantine** | Separar registros que no cumplen reglas de calidad | Parquet | Conservación del registro rechazado y motivo del problema |

Los archivos de Landing permanecerán inmutables.

Parquet será utilizado como formato intermedio gestionado por Spark para Bronze, Silver, Gold y los registros de quarantine.

## 7.3 Estrategia de particionamiento

El particionamiento no será aplicado de la misma manera a todas las fuentes.

La decisión dependerá principalmente de:

- volumen;
- crecimiento esperado;
- comportamiento temporal;
- modalidad batch o streaming;
- forma prevista de consulta.

Se evitará particionar por columnas de alta cardinalidad, como identificadores individuales, ya que esto podría producir una gran cantidad de particiones pequeñas.

### Estrategia propuesta

| Fuente / tipo | Particionamiento propuesto | Justificación |
|---|---|---|
| Maestros pequeños (`customers_orgs`, `users`, `resources`) | Partición técnica por fecha de ingesta cuando sea necesario | Permite organizar cargas sucesivas sin crear particiones por cada entidad |
| `billing_monthly` | `month` | La fuente posee naturalmente un grano y procesamiento mensual |
| `support_tickets` | Fecha derivada de `created_at` cuando el volumen lo justifique | Facilita procesamiento y análisis temporal |
| `nps_surveys` | Fecha o período de encuesta cuando el volumen lo justifique | La fuente posee naturaleza temporal |
| `marketing_touches` | Fecha derivada de `timestamp` cuando el volumen lo justifique | Permite organizar interacciones por período |
| `usage_events` | Fecha derivada de `timestamp` | Es la fuente de mayor crecimiento y las consultas se realizan principalmente por períodos |
| Gold diarios | Fecha correspondiente al grano del mart | Facilita consultas por rangos de tiempo |
| Gold mensuales | Mes | Coincide con el grano temporal del producto analítico |

La cantidad de particiones deberá revisarse durante la implementación para evitar generar archivos excesivamente pequeños.

## 7.4 Convenciones de nombres

Para mantener consistencia y facilitar la reproducción del proyecto se utilizarán las siguientes convenciones:

- nombres en minúsculas;
- formato `snake_case`;
- sin espacios;
- sin caracteres especiales ni acentos;
- nombres descriptivos;
- conservación del nombre lógico de la fuente en Landing y Bronze;
- nombres orientados al negocio para los productos Gold.

### Ejemplos

Fuentes y capas intermedias:

- `customers_orgs`
- `support_tickets`
- `billing_monthly`
- `usage_events`

Productos Gold:

- `org_daily_usage_by_service`
- `revenue_by_org_month`
- `cost_anomaly_mart`
- `tickets_by_org_date`
- `genai_tokens_by_org_date`

Las convenciones deberán mantenerse tanto en carpetas como en tablas, columnas y artefactos relacionados cuando corresponda.

## 7.5 Metadatos y trazabilidad

A partir de Bronze se incorporarán metadatos técnicos que permitan reconstruir el origen de los datos.

Como mínimo se consideran:

### `source_file`

Identifica el archivo desde el cual se obtuvo el registro.

Permite:

- localizar el origen de un dato;
- investigar errores;
- reproducir cargas;
- auditar archivos procesados.

### `ingest_ts`

Registra el momento en que el dato fue incorporado al pipeline.

Permite:

- distinguir diferentes ejecuciones;
- conocer cuándo se procesó una fuente;
- facilitar auditoría y trazabilidad.

### `schema_version`

En `usage_events` se conservará la versión de esquema provista por la fuente.

Esto permite diferenciar eventos v1 y v2 y aplicar correctamente las reglas de compatibilización.

Los metadatos técnicos deberán conservarse durante las transformaciones cuando sean necesarios para mantener el linaje del dato.

## 7.6 Estrategia inicial de retención

La consigna requiere justificar una política de retención, pero no establece períodos de conservación específicos.

Para el alcance académico de este proyecto se propone conservar todos los datos utilizados durante las tres instancias de evaluación, debido a que el dataset es reducido y es necesario mantener reproducibilidad y trazabilidad entre entregas.

### Criterio inicial

| Zona | Retención propuesta para el proyecto |
|---|---|
| Landing | Conservar todos los archivos originales durante todo el proyecto |
| Bronze | Conservar las versiones necesarias para reproducir el procesamiento |
| Silver | Conservar los datos conformados utilizados por las salidas Gold |
| Gold | Conservar los productos analíticos necesarios para las consultas y demostraciones |
| Quarantine | Conservar los registros rechazados mientras sean necesarios para análisis y evidencia |

En un escenario productivo real, los períodos de retención deberían definirse considerando volumen, costos de almacenamiento, requisitos operativos, regulatorios y de auditoría.

La definición de períodos productivos concretos queda fuera del alcance actual debido a que esos requisitos no fueron especificados.

## 7.7 Reglas de promoción entre zonas

Un dato no debe avanzar automáticamente entre las zonas del Data Lake.

Cada promoción deberá cumplir criterios mínimos.

### Landing → Bronze

Un archivo puede ser procesado hacia Bronze cuando:

- puede ser leído correctamente;
- corresponde a una fuente conocida;
- puede aplicarse el esquema esperado;
- se incorporan los metadatos técnicos;
- se conserva el mismo grano de la fuente;
- se controla la duplicación cuando corresponda.

Los archivos originales permanecen sin modificaciones en Landing.

---

### Bronze → Silver

Los registros pueden ser promovidos a Silver cuando:

- los tipos pueden ser normalizados;
- las claves necesarias son válidas;
- se aplicaron las reglas de calidad definidas;
- las versiones de esquema fueron compatibilizadas;
- los nulos fueron interpretados según la semántica del campo;
- los datos necesarios para joins poseen relaciones válidas.

Los registros que incumplan reglas consideradas críticas podrán ser enviados a `quarantine` para su análisis.

---

### Silver → Gold

Los datos pueden utilizarse para construir Gold cuando:

- poseen calidad suficiente para el caso de uso;
- las dimensiones necesarias ya fueron conformadas;
- las métricas pueden calcularse de manera reproducible;
- el grano del producto analítico está claramente definido;
- el resultado responde a una pregunta de negocio concreta.

Gold no representa una copia adicional de Silver, sino información preparada específicamente para consumo.

## 7.8 Aplicación de las reglas al dataset analizado

El perfil inicial realizado en el Punto 3 permite anticipar cómo se aplicarán las reglas de promoción.

| Situación observada | Tratamiento previsto |
|---|---|
| `value` llega algunas veces como texto | Conservar en Landing y aplicar casting controlado para Silver |
| `value` posee valores nulos | Evaluar la semántica antes de promover |
| `unit` posee valores nulos | Validar si corresponde según la métrica |
| `carbon_kg` no existe en schema v1 | Compatibilizar esquemas sin considerar automáticamente esos nulos como error |
| `genai_tokens` sólo aplica en determinados eventos | Validar según versión de esquema y servicio |
| Costos negativos | Aplicar regla de calidad y flag de anomalía |
| `event_id` actualmente no presenta duplicados | Mantener deduplicación preventiva para streaming y reprocesamiento |
| `org_id` y `resource_id` poseen integridad referencial | Permitir joins de enriquecimiento en Silver |

Las reglas definitivas serán implementadas y probadas en las siguientes etapas del proyecto.

## 7.9 Resumen del diseño del Data Lake

La estrategia propuesta puede resumirse de la siguiente manera:

| Aspecto | Decisión |
|---|---|
| Zonas principales | Landing, Bronze, Silver y Gold |
| Área auxiliar | Quarantine |
| Landing | Formatos originales inmutables |
| Formato intermedio | Parquet |
| Particionamiento | Principalmente temporal y según características de cada fuente |
| Naming | Minúsculas y `snake_case` |
| Metadatos mínimos | `source_file`, `ingest_ts` y `schema_version` cuando corresponda |
| Calidad | Validaciones antes de promover entre capas |
| Retención académica | Conservación de los datos necesarios durante todo el proyecto |
| Trazabilidad | Desde Landing hasta las salidas Gold |
| Consumo | Gold → Cassandra/AstraDB → dominios analíticos |

El diseño busca mantener una separación clara entre datos originales, datos técnicamente estandarizados, datos confiables y productos analíticos, evitando transformaciones innecesarias en las primeras capas.

# 8 - Flujo de datos batch y streaming

La arquitectura propuesta combina dos formas de procesamiento:

- **batch**, para maestros, facturación y otras fuentes periódicas;
- **streaming**, para eventos de utilización que requieren procesamiento incremental.

Ambos caminos comienzan en Landing y utilizan Apache Spark como motor de procesamiento.

La diferencia principal se encuentra en la forma de ingestión y en el tratamiento inicial de los datos. Posteriormente, ambos flujos pueden converger en Silver y contribuir a los productos analíticos de Gold.

En esta primera evaluación se diseña el flujo. La implementación end-to-end se realizará progresivamente en las siguientes entregas.

## 8.1 Flujo batch

El flujo batch se utilizará principalmente para:

- `customers_orgs.csv`;
- `users.csv`;
- `resources.csv`;
- `support_tickets.csv`;
- `marketing_touches.csv`;
- `nps_surveys.csv`;
- `billing_monthly.csv`.

El recorrido conceptual será:

```text
Archivo CSV
    │
    ▼
LANDING
Archivo original e inmutable
    │
    ▼
PySpark
spark.read
    │
    ▼
Validación del esquema
+ metadatos técnicos
    │
    ▼
BRONZE
Parquet
    │
    ▼
PySpark
limpieza / normalización / joins
    │
    ▼
SILVER
Parquet conformado
    │
    ▼
PySpark
agregaciones / métricas
    │
    ▼
GOLD
Marts analíticos
    │
    ▼
Cassandra / AstraDB
    │
    ▼
Consumo analítico
```

En Bronze se conservará el grano de la fuente y se incorporarán progresivamente esquemas explícitos, `source_file` e `ingest_ts`.

En Silver se aplicarán las transformaciones necesarias para obtener datos confiables y conformados.

En Gold se construirán productos de datos orientados a las consultas del negocio.

## 8.2 Ejemplo de flujo batch: facturación mensual

Se toma `billing_monthly.csv` como ejemplo de una fuente batch.

```text
billing_monthly.csv
        │
        ▼
     LANDING
        │
        ▼
 PySpark DataFrame
        │
        ├── esquema explícito
        ├── source_file
        └── ingest_ts
        │
        ▼
     BRONZE
   Parquet / month
        │
        ▼
      SILVER
        │
        ├── tipos numéricos consistentes
        ├── tratamiento de credits
        ├── validación de monedas
        └── relación con org_id
        │
        ▼
       GOLD
 revenue_by_org_month
        │
        ▼
Cassandra / AstraDB
```

La fuente posee un comportamiento naturalmente mensual, por lo que `month` constituye una partición coherente para este caso.

El producto Gold previsto deberá permitir posteriormente analizar revenue, créditos, impuestos y normalización monetaria por organización y mes.

## 8.3 Flujo streaming

El flujo streaming se utilizará para:

`usage_events_stream/*.jsonl`

Los archivos JSONL representan micro-lotes simulados de eventos de utilización.

El recorrido conceptual será:

```text
Archivos JSONL
      │
      ▼
   LANDING
      │
      ▼
Structured Streaming
spark.readStream
      │
      ├── esquema explícito
      ├── watermark
      ├── deduplicación por event_id
      ├── manejo de late data
      └── checkpoint
      │
      ▼
    BRONZE
Parquet particionado
      │
      ▼
    SILVER
      │
      ├── casting controlado
      ├── compatibilidad v1 / v2
      ├── reglas de calidad
      ├── joins con dimensiones
      └── tratamiento de anomalías
      │
      ▼
     GOLD
métricas por organización,
fecha y servicio
      │
      ▼
Cassandra / AstraDB
      │
      ▼
Consumo near real-time
```

La fuente provista es estática, pero su fragmentación permite simular el ingreso de micro-lotes. Durante la implementación, Structured Streaming será utilizado para procesar incrementalmente esos archivos.

## 8.4 Ejemplo de flujo streaming: evento de utilización

Un evento individual puede recorrer conceptualmente las siguientes etapas:

```text
JSONL original
        │
        ▼
LANDING
value = "15.7"
schema_version = 2
event_id = evt_123
        │
        ▼
BRONZE
mismo evento
+ source_file
+ ingest_ts
        │
        ▼
SILVER
value = 15.7
tipos normalizados
schema v1/v2 compatible
organización y recurso enriquecidos
calidad validada
        │
        ▼
GOLD
org_id
usage_date
service
daily_cost_usd
requests
otras métricas
```

El cambio de capa no implica únicamente mover archivos.

Cada promoción incorpora un nivel adicional de estructura, calidad o valor analítico.

## 8.5 Comparación entre los flujos

| Característica | Batch | Streaming |
|---|---|---|
| Fuentes principales | Maestros, soporte, marketing, NPS, facturación | Usage events |
| Entrada | CSV | JSONL |
| Lectura Spark prevista | `spark.read` | `spark.readStream` |
| Forma de procesamiento | Ejecuciones periódicas | Micro-lotes incrementales |
| Esquema explícito | Sí | Sí |
| Metadatos técnicos | `source_file`, `ingest_ts` | `source_file`, `ingest_ts` |
| Deduplicación | Cuando corresponda | `event_id` |
| Watermark | No requerido para fuentes actuales | Sí |
| Checkpoint | No requerido como estado de streaming | Sí |
| Salida intermedia | Parquet | Parquet |
| Convergencia | Silver / Gold | Silver / Gold |

La utilización de dos mecanismos de ingesta responde a las características reales de las fuentes y constituye la aplicación concreta del patrón Lambda seleccionado previamente.

## 8.6 Herramientas utilizadas en cada etapa

| Etapa | Herramienta / tecnología | Uso previsto |
|---|---|---|
| Entorno de ejecución | Google Colab | Desarrollo y ejecución reproducible |
| Procesamiento | Apache Spark / PySpark | Transformaciones batch y streaming |
| Batch | Spark DataFrames | Lectura y transformación de CSV |
| Streaming | Structured Streaming | Procesamiento incremental de JSONL |
| Landing | CSV / JSONL | Conservación de formatos originales |
| Bronze | Parquet | Datos crudos estandarizados |
| Silver | Parquet | Datos conformados |
| Gold | Parquet | Marts analíticos |
| Serving | Cassandra / AstraDB | Publicación orientada a consultas |

La selección mantiene el proyecto dentro de las tecnologías trabajadas durante la cursada y de aquellas requeridas explícitamente por la consigna.

## 8.7 Conclusión de los flujos

La solución utiliza dos caminos de ingestión diferenciados:

- batch para fuentes maestras y periódicas;
- streaming para eventos de utilización.

Ambos caminos conservan los datos originales en Landing, generan representaciones estandarizadas en Bronze y convergen progresivamente en Silver y Gold.

Esta separación permite atender simultáneamente necesidades de procesamiento periódico y near real-time sin forzar todas las fuentes a utilizar el mismo mecanismo de ingesta.

La implementación técnica completa de estos flujos se realizará durante las siguientes etapas del proyecto.

# 9 - Flujo batch de referencia mediante lógica MapReduce

MapReduce permite expresar un procesamiento distribuido separándolo conceptualmente en etapas.

Para este proyecto se utilizará como ejemplo el cálculo del:

**costo diario por organización y servicio**

a partir de un conjunto histórico de eventos de utilización procesado en modalidad batch.

El resultado tendrá como grano:

`org_id + usage_date + service`

y permitirá obtener el costo acumulado correspondiente a cada combinación.

Este ejemplo se utiliza para representar conceptualmente las etapas **Map → Shuffle → Reduce** y posteriormente mostrar su equivalente mediante DataFrames de PySpark.

## 9.1 Representación conceptual MapReduce

### MAP

Cada evento de utilización se transforma conceptualmente en un par:

```text
clave = (org_id, usage_date, service)
valor = cost_usd_increment
```

Por ejemplo:

```text
Evento 1
org_id = org_01
fecha = 2025-07-03
service = compute
cost = 2.50

        ↓ MAP

((org_01, 2025-07-03, compute), 2.50)
```

Si existen varios eventos:

```text
((org_01, 2025-07-03, compute), 2.50)
((org_01, 2025-07-03, compute), 1.80)
((org_01, 2025-07-03, storage), 0.90)
((org_02, 2025-07-03, compute), 4.20)
```

cada evento puede ser procesado inicialmente de forma independiente.

---

### SHUFFLE / GROUPING

Los pares generados son redistribuidos para reunir los registros que poseen la misma clave.

```text
(org_01, 2025-07-03, compute)
    ├── 2.50
    └── 1.80

(org_01, 2025-07-03, storage)
    └── 0.90

(org_02, 2025-07-03, compute)
    └── 4.20
```

Esta etapa permite que los valores asociados a una misma organización, fecha y servicio queden disponibles para la misma agregación.

---

### REDUCE

Finalmente se aplica una función de agregación sobre cada grupo.

En este caso:

```text
SUM(cost_usd_increment)
```

Resultado:

```text
(org_01, 2025-07-03, compute) → 4.30
(org_01, 2025-07-03, storage) → 0.90
(org_02, 2025-07-03, compute) → 4.20
```

El resultado representa el costo diario acumulado por organización y servicio.

## 9.2 Equivalente mediante PySpark DataFrames

Apache Spark permite expresar la misma lógica utilizando operaciones de alto nivel sobre DataFrames.

Para esta demostración se reutilizará el DataFrame `events` cargado previamente en modo batch.

Se derivará la fecha del evento y luego se agrupará por:

- `org_id`;
- `usage_date`;
- `service`.

Finalmente se sumará `cost_usd_increment`.

Este cálculo es una demostración del mecanismo de procesamiento y no constituye todavía el mart Gold definitivo, ya que en una implementación completa los datos deberán provenir previamente de Silver y haber superado las reglas de calidad correspondientes.

In [ ]:
from pyspark.sql import functions as F

# Preparación mínima para el ejemplo batch
events_batch_demo = (
    events
    .withColumn(
        "usage_date",
        F.to_date(F.to_timestamp("timestamp"))
    )
    .filter(
        F.col("cost_usd_increment").isNotNull()
    )
)

# Agregación equivalente a la lógica MapReduce
daily_cost_demo = (
    events_batch_demo
    .groupBy(
        "org_id",
        "usage_date",
        "service"
    )
    .agg(
        F.round(
            F.sum("cost_usd_increment"),
            4
        ).alias("daily_cost_usd"),
        F.count("*").alias("cantidad_eventos")
    )
    .orderBy(
        "usage_date",
        "org_id",
        "service"
    )
)

daily_cost_demo.show(
    20,
    truncate=False
)

## 9.3 Correspondencia MapReduce → PySpark

| Lógica MapReduce | Operación PySpark |
|---|---|
| Leer cada registro | DataFrame `events` |
| MAP: obtener clave y valor | Selección de `org_id`, `usage_date`, `service` y `cost_usd_increment` |
| SHUFFLE / agrupamiento | `groupBy("org_id", "usage_date", "service")` |
| REDUCE | `sum("cost_usd_increment")` |
| Resultado | `daily_cost_usd` por organización, fecha y servicio |

La API de DataFrames permite expresar la transformación sin implementar manualmente un `Mapper` y un `Reducer`.

Spark se encarga de construir y ejecutar el plan necesario para realizar el agrupamiento y la agregación de manera distribuible.

In [ ]:
eventos_entrada = events_batch_demo.count()
grupos_salida = daily_cost_demo.count()

print("Eventos de entrada:", eventos_entrada)
print("Grupos resultantes:", grupos_salida)

## 9.4 Relación con el producto analítico del proyecto

El procesamiento utilizado como ejemplo se relaciona directamente con el mart previsto:

`org_daily_usage_by_service`

cuyo grano es:

`org_id + usage_date + service`

y que deberá contener métricas de utilización, requests y costos diarios.

En una implementación posterior, el mart no se calculará directamente a partir de Landing.

El flujo esperado será:

**Landing → Bronze → Silver → Gold**

De esta manera, las métricas Gold se calcularán sobre información previamente tipificada, validada y conformada.

El ejemplo actual tiene como objetivo demostrar la lógica del procesamiento batch y su correspondencia conceptual con MapReduce.

## 9.5 Conclusión del procesamiento batch

El ejemplo permite representar un problema del proyecto mediante las etapas conceptuales de MapReduce:

1. **Map:** generar una clave compuesta por organización, fecha y servicio junto con el costo del evento.
2. **Shuffle:** reunir los eventos que comparten la misma clave.
3. **Reduce:** sumar los costos correspondientes a cada grupo.

PySpark permite expresar esta lógica mediante DataFrames utilizando `groupBy()` y `agg()`.

De esta forma se conserva la relación conceptual con MapReduce mientras se utiliza Apache Spark como motor de procesamiento principal del proyecto.

# 10 - Supuestos, riesgos, mitigaciones y decisiones abiertas

El diseño realizado hasta este punto parte de determinadas condiciones conocidas y de otras que todavía deberán validarse durante la implementación.

Por este motivo se documentan:

- supuestos utilizados para diseñar la solución;
- riesgos técnicos y de calidad;
- mitigaciones previstas;
- decisiones que todavía deberán cerrarse en las siguientes etapas.

El objetivo no es eliminar toda incertidumbre en esta primera entrega, sino identificarla explícitamente para evitar que se transforme posteriormente en problemas no previstos.

## 10.1 Supuestos iniciales

Para el diseño de esta primera versión de la arquitectura se adoptan los siguientes supuestos:

1. Los archivos provistos representan correctamente las fuentes que utilizará el proyecto académico.

2. Los datos almacenados en Landing deben conservarse sin modificaciones.

3. `org_id`, `resource_id`, `event_id` y las demás claves candidatas mantienen el significado observado durante el perfil inicial.

4. Las fuentes maestras y periódicas serán procesadas principalmente mediante batch.

5. `usage_events_stream` representa la fuente destinada al procesamiento mediante Structured Streaming.

6. La fragmentación de los archivos JSONL representa micro-lotes simulados y no una tasa real de llegada productiva.

7. Parquet será el formato intermedio utilizado por Spark para Bronze, Silver y Gold.

8. Cassandra/AstraDB será utilizado posteriormente como capa de serving.

9. El dataset académico es suficientemente pequeño para ser procesado en Google Colab, aunque la arquitectura se diseña considerando crecimiento futuro.

10. Los requisitos de retención productiva, frecuencia exacta de algunas fuentes batch y niveles de servicio no fueron especificados, por lo que permanecerán como decisiones abiertas.

## 10.2 Riesgos y mitigaciones

| Riesgo | Evidencia / causa | Impacto potencial | Mitigación prevista |
|---|---|---|---|
| Evolución de esquema | Existen schema v1 y v2 | Transformaciones incompatibles o pérdida de información | Compatibilizar versiones en Silver y conservar `schema_version` |
| Tipos ambiguos | `value` puede llegar como número o texto | Errores en cálculos y agregaciones | Esquema explícito y casting controlado |
| Valores nulos | Se observaron nulos en varias fuentes | Métricas incompletas o interpretaciones incorrectas | Reglas específicas según la semántica de cada campo |
| Costos negativos | Se detectaron eventos con costos negativos | Distorsión de métricas FinOps | Validación y flag de anomalía antes de construir Gold |
| Outliers o spikes | La consigna anticipa valores atípicos | Promedios y acumulaciones distorsionadas | Perfil estadístico y método de detección justificado |
| Duplicados futuros | Actualmente no existen duplicados de `event_id`, pero pueden aparecer por reprocesamiento | Doble contabilización | Deduplicación por clave natural y diseño idempotente |
| Late data | Los eventos pueden llegar fuera del tiempo esperado | Métricas temporales incompletas | Watermark y estrategia de manejo de eventos tardíos |
| Pérdida de estado del streaming | Una ejecución puede interrumpirse | Reprocesamiento o pérdida de continuidad | Checkpointing |
| Registros inválidos | Algunas reglas de calidad pueden fallar | Datos no confiables en Silver o Gold | Separación en `quarantine` con motivo del rechazo |
| Inconsistencias referenciales futuras | Nuevos eventos podrían utilizar claves inexistentes | Fallos o pérdida de información en joins | Validación de `org_id` y `resource_id` antes del enriquecimiento |
| Exceso de particiones pequeñas | Un particionamiento demasiado granular puede generar muchos archivos | Menor eficiencia de procesamiento | Particionar sólo cuando esté justificado y revisar `repartition` / `coalesce` durante implementación |
| Credenciales en el repositorio | Cassandra/AstraDB requerirá configuración de acceso | Riesgo de seguridad | Configuración externa y exclusión de secretos del repositorio |
| Diferencias entre diseño e implementación | La arquitectura puede evolucionar durante las siguientes entregas | Documentación inconsistente | Actualizar diagramas y decisiones junto con la implementación |

## 10.3 Decisiones todavía abiertas

Algunas decisiones deberán cerrarse durante la implementación, cuando exista evidencia técnica suficiente.

### Watermark de streaming

Se deberá definir cuánto retraso será aceptado para los eventos tardíos.

La consigna exige utilizar watermark, pero no define un umbral concreto. Por lo tanto, no se fija todavía un valor arbitrario.

---

### Umbral para detección de anomalías

La solución deberá definir posteriormente un criterio para identificar costos atípicos.

Entre las alternativas contempladas por la consigna se encuentran métodos como z-score, MAD o percentiles.

La selección y el umbral se definirán luego de analizar la distribución real de los costos.

---

### Cantidad de particiones Spark

Durante esta primera etapa se definió una estrategia lógica de particionamiento.

La cantidad concreta de particiones y el uso de `repartition()` o `coalesce()` deberán evaluarse utilizando tamaños y tiempos observados durante la implementación.

---

### Frecuencia exacta de las cargas batch

La facturación posee una naturaleza mensual, pero la periodicidad de actualización de todas las demás fuentes batch no está completamente especificada.

La frecuencia deberá definirse en función de los requisitos operativos que se establezcan posteriormente.

---

### Retención productiva

Para el proyecto académico se conservarán los datos necesarios para mantener reproducibilidad.

En un entorno productivo, los períodos concretos deberán definirse según volumen, costos y necesidades de auditoría.

---

### Modelo físico de Cassandra/AstraDB

En esta etapa se conocen las consultas objetivo, pero el diseño definitivo de las tablas query-first se realizará en las siguientes entregas.

## 10.4 Priorización inicial

Para orientar las siguientes etapas se propone la siguiente prioridad:

| Prioridad | Riesgos principales |
|---|---|
| **Alta** | Evolución de esquema, tipos ambiguos, duplicación, calidad de costos y reproducibilidad |
| **Media** | Late data, nulos según contexto, integridad referencial, particionamiento |
| **Baja en la primera etapa** | Optimización avanzada de rendimiento y política productiva de retención |

La prioridad podrá modificarse a partir de los resultados obtenidos durante la implementación.

## 10.5 Conclusión de riesgos y decisiones

Los principales riesgos identificados están relacionados con:

- calidad y consistencia de los datos;
- evolución del esquema;
- procesamiento incremental;
- idempotencia;
- particionamiento;
- seguridad;
- reproducibilidad.

Para cada riesgo se definió una estrategia inicial de mitigación.

Las decisiones que todavía requieren evidencia de implementación se mantienen explícitamente abiertas en lugar de fijar valores arbitrarios.

Este registro deberá actualizarse durante las siguientes evaluaciones a medida que la arquitectura conceptual se transforme en un pipeline ejecutable.

# 11 - Estimación preliminar de esfuerzo, roles y recursos

La implementación del proyecto se realizará de manera incremental a lo largo de las tres instancias de evaluación.

En esta primera etapa no se busca dimensionar una infraestructura productiva definitiva, sino identificar las actividades principales, los roles necesarios y los recursos técnicos previstos para completar el proyecto de forma reproducible.

Los roles representan responsabilidades lógicas. En un equipo pequeño o en un trabajo individual, una misma persona puede asumir más de un rol.

## 11.1 Roles principales

| Rol | Responsabilidades principales |
|---|---|
| **Data Engineer** | Ingesta, PySpark, Structured Streaming, Data Lake, Parquet, calidad, joins y construcción de pipelines |
| **Data Analyst / Data Scientist** | Comprensión de las fuentes, definición de métricas, análisis de calidad y validación de productos Gold |
| **Arquitectura / Gobierno de datos** | Diseño de capas, decisiones técnicas, trazabilidad, seguridad, metadatos y documentación |
| **Responsable de documentación y pruebas** | README, evidencias, pruebas de ejecución, registro de decisiones y preparación de la defensa |

Se realizar el proyecto con una sola persona. Estas responsabilidades serán asumidas por el mismo integrante en diferentes momentos del desarrollo.

## 11.2 Estimación preliminar del esfuerzo

Se propone distribuir el esfuerzo de manera progresiva:

| Etapa | Actividades principales | Esfuerzo relativo estimado |
|---|---|---:|
| **Evaluación 1 — Diseño y fundación** | Comprensión, 5V, perfil de datos, arquitectura, Data Lake, flujos y documentación | 25 % |
| **Evaluación 2 — Implementación técnica** | Bronze, streaming, Silver, calidad, Gold inicial, Cassandra/AstraDB y pruebas | 45 % |
| **Evaluación final — Integración y cierre** | Completar dominios, consultas, gobierno, pruebas, documentación y defensa | 30 % |

La estimación es preliminar y podrá modificarse según los resultados obtenidos durante la implementación.

La mayor carga técnica se espera durante la segunda evaluación, debido a que allí la propuesta conceptual deberá transformarse en un pipeline ejecutable.

## 11.3 Recursos técnicos previstos

| Recurso | Utilización |
|---|---|
| **Google Colab** | Entorno principal de desarrollo y ejecución |
| **Python** | Lenguaje utilizado junto con PySpark |
| **Apache Spark / PySpark** | Procesamiento batch y transformaciones |
| **Structured Streaming** | Procesamiento incremental de usage events |
| **Parquet** | Formato intermedio del Data Lake |
| **Cassandra / AstraDB** | Serving analítico en etapas posteriores |
| **GitHub** | Versionado del notebook, documentación y artefactos |
| **Dataset Cloud Provider Analytics** | Fuentes de datos del proyecto |

Para la primera evaluación no se requiere infraestructura adicional a la necesaria para explorar los datos y documentar la arquitectura propuesta.

## 11.4 Conclusión de esfuerzo y recursos

El proyecto puede desarrollarse utilizando las herramientas definidas en la cursada y en la consigna, sin incorporar infraestructura adicional en esta primera etapa.

La estrategia será priorizar:

1. una fundación arquitectónica coherente;
2. una implementación mínima y reproducible;
3. validación de calidad y resultados;
4. documentación consistente con la solución real;
5. optimización únicamente cuando exista evidencia que la justifique.

La estimación deberá revisarse después del feedback de la primera evaluación y durante la implementación técnica.

# 12 - Repositorio inicial, convenciones y evidencias

El proyecto se mantendrá en un repositorio versionado para conservar:

- el notebook ejecutable;
- la documentación;
- el diagrama de arquitectura;
- las decisiones técnicas;
- las evidencias de ejecución;
- las instrucciones necesarias para reproducir el análisis.

Google Colab será el entorno principal de desarrollo del notebook, mientras que GitHub permitirá mantener una versión identificable y reproducible de los artefactos entregados.

## 12.1 Estructura inicial del repositorio

Se propone comenzar con la siguiente estructura:

```text
cloud-provider-analytics/
│
├── README.md
├── DECISIONS.md
│
├── notebooks/
│   └── 01_evaluacion_1_fundacion.ipynb
│
├── docs/
│   ├── evaluacion_1.pdf
│   └── arquitectura_v1.png
│
├── data/
│   └── README.md
│
└── evidence/
    └── README.md
```

La estructura podrá ampliarse durante las siguientes entregas con carpetas para código productivo, configuración, pruebas e infraestructura cuando dichas necesidades aparezcan realmente.

## 12.2 Contenido previsto de cada artefacto

### `README.md`

Contendrá:

- objetivo del proyecto;
- contexto;
- arquitectura propuesta;
- tecnologías utilizadas;
- estructura del repositorio;
- instrucciones para ejecutar el notebook;
- ubicación de los datos;
- alcance y limitaciones actuales.

### `DECISIONS.md`

Registrará decisiones relevantes, por ejemplo:

- selección del patrón Lambda;
- utilización de Parquet;
- estrategia de particionamiento;
- reglas de promoción entre capas;
- decisiones todavía abiertas.

### `notebooks/`

Contendrá el notebook de Google Colab utilizado para la evaluación y las futuras etapas del proyecto.

### `docs/`

Contendrá los artefactos de diseño:

- documento de evaluación;
- diagrama de arquitectura;
- documentación adicional cuando corresponda.

### `data/README.md`

No será necesario duplicar todo el dataset dentro del repositorio.

El archivo documentará cómo obtener o cargar el dataset requerido para ejecutar el notebook.

### `evidence/`

Permitirá almacenar evidencias relevantes de ejecución cuando sea necesario, como capturas, resultados o logs.

## 12.3 Evidencia mínima generada en esta evaluación

El notebook contiene evidencia reproducible de exploración del dataset.

Entre los resultados generados se encuentran:

| Evidencia | Resultado |
|---|---|
| Fuentes identificadas | 8 fuentes lógicas |
| Eventos de usage | 43.200 |
| Archivos JSONL | 120 |
| Versiones de esquema | 2 |
| Schema v1 | 10.800 eventos |
| Schema v2 | 32.400 eventos |
| Duplicados de claves principales | 0 detectados |
| `org_id` sin organización maestra | 0 |
| `resource_id` de eventos sin maestro | 0 |
| `value` nulo | 877 |
| `unit` nulo | 2.075 |
| Costos negativos | 216 |
| Valores de `value` originalmente almacenados como texto | 1.309 |

Además se generó evidencia mediante:

- impresión de esquemas;
- conteos;
- análisis de nulos;
- análisis de duplicados;
- integridad referencial;
- cobertura temporal;
- trazabilidad mediante archivo de origen;
- ejemplo de procesamiento batch mediante `groupBy()` y agregación.

## 12.4 Convenciones del proyecto

Se utilizarán las siguientes convenciones:

- nombres descriptivos;
- minúsculas y `snake_case`;
- notebooks numerados según el orden de desarrollo;
- separación entre documentación, datos, código y evidencias;
- comentarios destinados a explicar decisiones y no únicamente repetir el código;
- ninguna credencial, token o secreto almacenado en el repositorio;
- los resultados deberán poder regenerarse a partir de instrucciones documentadas.

Los cambios posteriores a cada entrega deberán mantenerse versionados para distinguir la versión evaluada de las correcciones posteriores.

# Verificación final de la primera evaluación

| Punto solicitado | Estado | Evidencia en el notebook |
|---|---|---|
| 1. Problema, usuarios, preguntas y objetivos | ✅ | Punto 1 |
| 2. Justificación mediante las 5V | ✅ | Punto 2 |
| 3. Inventario y perfil de fuentes | ✅ | Punto 3 |
| 4. Arquitectura de alto nivel | ✅ | Punto 4 |
| 5. Patrón arquitectónico | ✅ | Punto 5 — Lambda |
| 6. Matriz requisito-componente | ✅ | Punto 6 |
| 7. Diseño del Data Lake | ✅ | Punto 7 |
| 8. Flujos batch y streaming | ✅ | Punto 8 |
| 9. MapReduce o equivalente conceptual | ✅ | Punto 9 |
| 10. Supuestos, riesgos y mitigaciones | ✅ | Punto 10 |
| 11. Esfuerzo, roles y recursos | ✅ | Punto 11 |
| 12. Repositorio y evidencia inicial | ✅ | Punto 12 |

La primera evaluación deja definida la fundación conceptual y técnica sobre la cual se implementará el pipeline durante las siguientes etapas.

# Conclusión de la primera entrega

La exploración realizada permitió comprender las características principales del caso Cloud Provider Analytics y verificar las condiciones presentes en los datos provistos.

A partir de esa evidencia se diseñó una arquitectura basada en:

**Landing → Bronze → Silver → Gold → Serving**

utilizando PySpark como motor principal de procesamiento, Parquet como formato intermedio y una estrategia Lambda que diferencia las fuentes batch de los eventos destinados a Structured Streaming.

El análisis permitió además identificar problemas reales de calidad, evolución de esquema, riesgos y decisiones que deberán resolverse progresivamente durante la implementación.

La solución propuesta busca mantener coherencia entre los requisitos del negocio, las características de los datos y las herramientas utilizadas durante la cursada, dejando una base reproducible para continuar el desarrollo del proyecto.